In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kendalltau

# ------------------------------------------------------------------ #
# Reuses the ground-truth parsing and per-protein assembly of rmse_comparison_results_10seedens.ipynb /
# kendalltau_comparison_results_10seedens.ipynb:
# tag/flag/4-sigma masks were already folded into each prediction's own mask when predictions were generated
# (checked: every tag and N_flag position is False in the mask, and the residue-level masks of all
# four model files are identical), so the manual 4-sigma BMRB outlier filter of the older
# correlation_ranking_offset.ipynb is no longer needed.
# ------------------------------------------------------------------ #
def parse_shift_list(lst, sequence, atom):
    vals, mask = [], []
    for i, x in enumerate(lst):
        residue = sequence[i]
        bad = False
        if x is None:
            bad = True
        elif isinstance(x, str):
            xs = x.strip().lower()
            if xs in {"none", "null", "nan", "inf", "+inf", "-inf"}:
                bad = True
            else:
                try:
                    x = float(x)
                except Exception:
                    bad = True
        if not bad and isinstance(x, (int, float)):
            if not math.isfinite(float(x)):
                bad = True
        if atom in ("N", "H") and residue == "P":
            bad = True
        if atom == "CB" and residue == "G":
            bad = True
        if bad:
            vals.append(0.0)
            mask.append(False)
        else:
            vals.append(float(x))
            mask.append(True)
    return vals, mask


def load_ground_truth(gt_path: Path, atom: str):
    """entry_ID (without the bmr prefix) -> (shift list, mask list)."""
    with open(gt_path) as f:
        rbmrb = json.load(f)
    out = {}
    for ref_id, shift_lst in rbmrb[atom].items():
        if all(x is None or (isinstance(x, str) and x.lower() in ("none", "null"))
               for x in shift_lst):
            continue
        sequence = rbmrb["sequence"][ref_id]
        shift, mask = parse_shift_list(shift_lst, sequence, atom=atom)
        entry = rbmrb["entry_ID"][ref_id].lower().removeprefix("bmr")
        out[entry] = (shift, mask)
    return out


def load_pred(path):
    with open(path) as f:
        d = json.load(f)
    return d["mean"], d["mask"]


def build_per_protein(pred_path, gt_path, atom):
    """entry -> {"pred": ..., "gt": ...}; intersects the prediction's own mask with the GT None/Pro-N-H mask only."""
    pred_dict, mask_dict = load_pred(pred_path)
    gt = load_ground_truth(Path(gt_path), atom=atom)

    per_protein = {}
    skipped = []
    for entry, (true_shift, true_mask) in gt.items():
        if entry not in pred_dict:
            skipped.append(entry)
            continue
        pred_shift = pred_dict[entry]
        pred_mask = mask_dict[entry]
        n = min(len(true_shift), len(true_mask), len(pred_shift), len(pred_mask))
        combined_mask = np.array(true_mask[:n], dtype=bool) & np.array(pred_mask[:n], dtype=bool)
        if not combined_mask.any():
            skipped.append(entry)
            continue
        per_protein[entry] = {
            "pred": np.asarray(pred_shift[:n], dtype=float)[combined_mask],
            "gt":   np.asarray(true_shift[:n], dtype=float)[combined_mask],
        }
    return per_protein, skipped


def load_offset_dict(offset_json_path, atom="N"):
    """pdb_id (without the bmr prefix) -> re-referencing offset (ppm)."""
    with open(offset_json_path) as f:
        d = json.load(f)
    out = {}
    for key, off in d["offset"].items():
        pdb_id = d["entry_ID"][key].lower().removeprefix("bmr")
        v = off.get(atom)
        if v is not None:
            out[pdb_id] = float(v)
    return out

RESULTS_DIR = Path("../../data/analysis/Fig2/results")
ATOM = "N"
MIN_VALID_RES = 3   # Kendall's tau is meaningless for very few residues; skip those proteins

OFFSET_JSON = {
    "panav": "../../data/rBMRB/bmrb_cs_data_withoffset_panav_v2.json",
    "lacs":  "../../data/rBMRB/bmrb_cs_data_withoffset_lacs_v2.json",
}
models_cfg = {"SLAE-CS-PANAV": "panav", "SLAE-CS-LACS": "lacs"}

# baseline: NOREF; as in kendalltau_comparison_results_10seedens.ipynb, the PANAV test set is the GT for every model
# (tau is insensitive to a constant per-protein referencing offset, so comparing against one GT frame is valid)
gt_path = RESULTS_DIR / "test_panav.json"
per_protein_noref, _ = build_per_protein(RESULTS_DIR / "predictions_N_seed_ensemble_10seed_noref.json",
                                          gt_path, ATOM)


def kendall_vs_noref(model_key):
    """For every protein with a non-None offset for this model, compute |offset|, the signed offset and
    dtau = tau(model) - tau(noref). tau ignores a constant per-protein shift, so it measures ranking ability
    rather than offset-dependent absolute accuracy (same rationale as the older correlation_ranking_offset.ipynb)."""
    per_protein_model, _ = build_per_protein(
        RESULTS_DIR / f"predictions_N_seed_ensemble_10seed_{model_key}.json", gt_path, ATOM)
    offset_dict = load_offset_dict(OFFSET_JSON[model_key], atom=ATOM)

    common = set(per_protein_model) & set(per_protein_noref) & set(offset_dict)
    sgn, absoff, dtau = [], [], []
    skipped = 0
    for entry in sorted(common):
        pred_m, gt_m = per_protein_model[entry]["pred"], per_protein_model[entry]["gt"]
        pred_n, gt_n = per_protein_noref[entry]["pred"], per_protein_noref[entry]["gt"]
        # The two models' masks should agree residue by residue (verified); check explicitly anyway so that
        # tau(model) and tau(noref) use exactly the same GT residues; skip the protein if they differ
        if gt_m.shape != gt_n.shape or not np.allclose(gt_m, gt_n):
            skipped += 1; continue
        if len(gt_m) < MIN_VALID_RES:
            skipped += 1; continue

        tau_m = kendalltau(pred_m, gt_m).correlation
        tau_n = kendalltau(pred_n, gt_n).correlation
        if np.isnan(tau_m) or np.isnan(tau_n):
            skipped += 1; continue

        off = offset_dict[entry]
        sgn.append(off); absoff.append(abs(off)); dtau.append(tau_m - tau_n)

    return np.array(sgn), np.array(absoff), np.array(dtau), skipped


In [ ]:
from scipy.stats import pearsonr, spearmanr, norm

def equivalence_report(x, y, label, sesoi=0.10, n_boot=2000, n_perm=2000, seed=0):
    """x, y: arrays of equal length (e.g. absoff / sgn and dtau). sesoi: smallest effect size of interest;
    a correlation with |r| below it is treated as practically negligible (0.1 is the usual "small effect" threshold;
    adjust as needed)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)
    print(f"\n=== {label}  (n={n}) ===")

    r_p, p_p = pearsonr(x, y)
    r_s, p_s = spearmanr(x, y)
    print(f"Pearson  r = {r_p:+.4f}  (p = {p_p:.3g})")
    print(f"Spearman rho = {r_s:+.4f}  (p = {p_s:.3g})")

    # ---- Fisher-z CI for Pearson r ----
    z = np.arctanh(r_p)
    se = 1 / np.sqrt(n - 3)
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    print(f"95% CI (Fisher-z) for r: [{lo:+.4f}, {hi:+.4f}]")

    # ---- TOST equivalence test: is |r| significantly SMALLER than sesoi? ----
    # H0: r <= -sesoi or r >= +sesoi (a "real" correlation exists)
    # H1: -sesoi < r < +sesoi        (r is equivalent to 0 within sesoi)
    # two one-sided z-tests on the Fisher-z scale, reject H0 (=> conclude equivalence)
    # only if BOTH one-sided tests reject their respective bound
    z_sesoi = np.arctanh(sesoi)
    z_lo = (z - (-z_sesoi)) / se     # test r > -sesoi
    z_hi = (z_sesoi - z) / se        # test r <  sesoi
    p_tost = 1 - norm.cdf(min(z_lo, z_hi))
    verdict = "EQUIVALENT to 0 (|r| < sesoi)" if p_tost < 0.05 else "not conclusively equivalent"
    print(f"TOST equivalence (SESOI=|r|<{sesoi}): p = {p_tost:.3g}  -> {verdict}")

    # ---- bootstrap CI (resample pairs) as a distribution-free cross-check ----
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    boot_r = []
    for _ in range(n_boot):
        b = rng.choice(idx, n, replace=True)
        if np.std(x[b]) == 0 or np.std(y[b]) == 0:
            continue
        boot_r.append(pearsonr(x[b], y[b])[0])
    boot_lo, boot_hi = np.percentile(boot_r, [2.5, 97.5])
    print(f"Bootstrap 95% CI for r (pair resampling, {len(boot_r)} draws): [{boot_lo:+.4f}, {boot_hi:+.4f}]")

    # ---- permutation null for Pearson r ----
    perm_r = np.empty(n_perm)
    for i in range(n_perm):
        perm_r[i] = pearsonr(x, rng.permutation(y))[0]
    p_perm = (np.abs(perm_r) >= abs(r_p)).mean()
    print(f"Permutation p-value (observed r vs {n_perm} shuffles): {p_perm:.3g}  "
          f"(null |r| mean={np.abs(perm_r).mean():.4f})")

    return dict(n=n, r=r_p, p_pearson=p_p, rho=r_s, p_spearman=p_s,
                ci_fisher=(lo, hi), p_tost=p_tost, ci_boot=(boot_lo, boot_hi), p_perm=p_perm)


results = {}
for name, key in models_cfg.items():
    sgn, absoff, dtau, sk = kendall_vs_noref(key)
    results[(name, "signed")]  = equivalence_report(sgn,    dtau, f"{name}: signed offset vs Δτ")
    results[(name, "abs")]     = equivalence_report(absoff, dtau, f"{name}: |offset| vs Δτ")

In [ ]:
def bootstrap_pearson_ci(x, y, n_boot=2000, seed=0):
    """returns (r, ci_lo, ci_hi) via pair-resampling bootstrap on Pearson r."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    r = pearsonr(x, y)[0]
    rng = np.random.default_rng(seed)
    n = len(x)
    idx = np.arange(n)
    boot_r = []
    for _ in range(n_boot):
        b = rng.choice(idx, n, replace=True)
        if np.std(x[b]) == 0 or np.std(y[b]) == 0:
            continue
        boot_r.append(pearsonr(x[b], y[b])[0])
    lo, hi = np.percentile(boot_r, [2.5, 97.5])
    return r, lo, hi

def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base + 2,
        "axes.labelsize": base - 2,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })

set_style(base=19)

# one colour per model
color_map = {
    "slae-cs-panav": "#61AFEF",   # blue
    "slae-cs-lacs":  "#E06C75",   # red
}

for name, key in models_cfg.items():
    sgn, absoff, dtau, sk = kendall_vs_noref(key)
    good = dtau > 0                                  # still used in the printout below

    r, ci_lo, ci_hi = bootstrap_pearson_ci(absoff, dtau)
    p_pearson = pearsonr(absoff, dtau)[1]

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(absoff, dtau, s=60, alpha=0.8, color=color_map[name.lower()])
    ax.axhline(0, color="black", lw=1)


    ax.set_xlabel(f"rBMRB-{key.upper()} |$^{{15}}$N offset| (ppm)")
    ax.set_ylabel(f"Δ Kendall τ ({name} - SLAE-CS-BASELINE)")
    ax.set_xlim(-0.1, 3)
    ax.set_ylim(-0.05, 0.05)
    # ax.grid(alpha=0.3, linestyle="--")

    # extra legend entry: Pearson r + bootstrap 95% CI (|offset| vs Δτ)
    ax.plot([], [], " ",
            label=f"r = {r:+.3f}  95% CI [{ci_lo:+.3f}, {ci_hi:+.3f}]\n"
                  f"Pearson p = {p_pearson:.2g}")

    # ax.legend(frameon=False, loc="best")
    plt.tight_layout()
    plt.savefig(f"Figs/offset_vs_dtau_10seedens_{name.lower()}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"Figs/offset_vs_dtau_10seedens_{name.lower()}.svg", bbox_inches="tight")
    plt.show()

    print(f"[{name}] n={len(dtau)}  skipped={sk}  "
          f"mean Δτ={dtau.mean():+.4f}  median={np.median(dtau):+.4f}  "
          f"%better={100*good.mean():.0f}  "
          f"high-offset(|off|>1ppm) mean Δτ={dtau[absoff>1].mean():+.4f} (n={int((absoff>1).sum())})")
